In [17]:
from pathlib import Path
import json
from sentence_transformers import SentenceTransformer
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize

In [18]:
# Define project and data paths
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/raw"

fantasy_data_path = DATA_DIR / "generated_stories_fantasy.jsonl"
romance_data_path = DATA_DIR / "generated_stories_romance.jsonl"
litterary_fiction_data_path = DATA_DIR / "generated_stories_litterary_fiction.jsonl"

In [19]:
# Read JSONL file
with open(romance_data_path, 'r') as f:
    romance_stories = [json.loads(line) for line in f]

In [20]:
model = SentenceTransformer("all-MiniLM-L6-v2")
tokenizer = model.tokenizer

In [31]:
def embed_story(text, model, chunk_size=200):
    tokenizer = model.tokenizer
    max_len = model.max_seq_length
    
    # Hard safety check
    assert chunk_size < max_len
    
    # Tokenize without special tokens
    tokens = tokenizer.encode(text, add_special_tokens=False)
    
    embeddings = []
    
    for i in range(0, len(tokens), chunk_size):
        chunk_tokens = tokens[i:i+chunk_size]
        
        # Convert tokens back to text
        chunk_text = tokenizer.decode(chunk_tokens)
        
        # EXTRA SAFETY CHECK
        token_count = len(tokenizer.encode(chunk_text))
        if token_count > max_len:
            print("Chunk too long:", token_count)
            continue
        
        chunk_embedding = model.encode(chunk_text, convert_to_numpy=True)
        embeddings.append(chunk_embedding)
    
    return np.mean(embeddings, axis=0)



In [36]:
story1 = romance_stories[0]["story"]
story2 = romance_stories[2]["story"]

emb1 = embed_story(story1, model)
emb2 = embed_story(story2, model)

In [37]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity([emb1], [emb2])[0][0]

print("Similarity score:", similarity)

Similarity score: 0.791478
